In [ ]:
!pip install ultralytics

In [ ]:
cd /content/drive/MyDrive/Colab Notebooks/Yolo-cls-solar-plane

/content/drive/MyDrive/Colab Notebooks/Yolo-cls-solar-plane


In [ ]:
import torch
import torch.nn as nn
from ultralytics import YOLO

model = YOLO("yolo11n-cls.pt")

# โมดูล PyTorch จริงอยู่ที่ model.model
print(model.model)  # ดูสรุปสถาปัตยกรรม



ClassificationModel(
  (model): Sequential(
    (0): Conv(
      (conv): Conv2d(3, 16, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
      (bn): BatchNorm2d(16, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (act): SiLU(inplace=True)
    )
    (1): Conv(
      (conv): Conv2d(16, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
      (bn): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (act): SiLU(inplace=True)
    )
    (2): C3k2(
      (cv1): Conv(
        (conv): Conv2d(32, 32, kernel_size=(1, 1), stride=(1, 1), bias=False)
        (bn): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (act): SiLU(inplace=True)
      )
      (cv2): Conv(
        (conv): Conv2d(48, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
        (bn): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (act): SiLU(inplace=True)
      )
  

In [ ]:
old_head = model.model.model[-1]
print("Last head module:", old_head)  # Classify(...)

Last head module: Classify(
  (conv): Conv(
    (conv): Conv2d(256, 1280, kernel_size=(1, 1), stride=(1, 1), bias=False)
    (bn): BatchNorm2d(1280, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (act): SiLU(inplace=True)
  )
  (pool): AdaptiveAvgPool2d(output_size=1)
  (drop): Dropout(p=0.0, inplace=True)
  (linear): Linear(in_features=1280, out_features=1000, bias=True)
)


In [ ]:
in_features  = old_head.linear.in_features
NUM_CLASSES = 5  # << เป้าหมายของเราคือ 5 คลาส
print(f"features {in_features}")
print(f"class {NUM_CLASSES}")

features 1280
class 5


In [ ]:
class CustomMLPHead(nn.Module):
    """
    หัวจำแนกใหม่สำหรับ YOLO classification:
    - AdaptiveAvgPool2d(1) + Flatten
    - MLP 1 ชั้นกลาง + BN + ReLU + Dropout
    - Linear ออก 5 คลาส
    """
    def __init__(self, in_ch, num_classes=NUM_CLASSES, hidden=512, p_drop=0.2):
        super().__init__()
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.flat = nn.Flatten()
        self.fc = nn.Sequential(
            nn.Linear(in_ch, hidden, bias=True),
            nn.BatchNorm1d(hidden),
            nn.ReLU(inplace=True),
            nn.Dropout(p_drop),
            nn.Linear(hidden, num_classes, bias=True),
        )
        # initialization เหมาะกับ ReLU
        for m in self.fc:
            if isinstance(m, nn.Linear):
                nn.init.kaiming_normal_(m.weight, nonlinearity='relu')
                nn.init.zeros_(m.bias)

    def forward(self, x):
        x = self.pool(x)   # (N, C, 1, 1)
        x = self.flat(x)   # (N, C)
        x = self.fc(x)     # (N, num_classes)
        return x

In [ ]:
# สร้างหัวใหม่ แล้วใส่แทนที่หัวเดิม
new_head = CustomMLPHead(in_ch = in_features, num_classes=NUM_CLASSES, hidden=512, p_drop=0.2)
model.model.model[-1] = new_head

# (ออปชัน) freeze backbone เทรนเฉพาะหัวใหม่ให้ติดก่อน
for p in model.model.model[:-1].parameters():
    p.requires_grad = False

print("New head:", model.model.model[-1])

New head: CustomMLPHead(
  (pool): AdaptiveAvgPool2d(output_size=1)
  (flat): Flatten(start_dim=1, end_dim=-1)
  (fc): Sequential(
    (0): Linear(in_features=1280, out_features=512, bias=True)
    (1): BatchNorm1d(512, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU(inplace=True)
    (3): Dropout(p=0.2, inplace=False)
    (4): Linear(in_features=512, out_features=5, bias=True)
  )
)


In [ ]:
from pathlib import Path
DATA_DIR = "Faulty_solar_panel_split"
assert (Path(DATA_DIR)/"train").exists(), "ตรวจ path DATA_DIR ให้ถูกต้อง"

model.train(
    data=DATA_DIR,
    epochs=10,
    imgsz=224,
    batch=64,
    optimizer="Adam",
    lr0=1e-3,
    momentum=0.9,
    weight_decay=5e-4
)

Ultralytics 8.3.214 🚀 Python-3.12.12 torch-2.8.0+cu126 CUDA:0 (Tesla T4, 15095MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=64, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=Faulty_solar_panel_split, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=10, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=224, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.001, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11n-cls.pt, momentum=0.9, mosaic=1.0, multi_scale=False, name=train10, nbs=64, nms=False, opset=None, optimize=False, optimizer=Adam, overlap_mask=True, patience=100, perspective=0.0, plots=True, pose=

ultralytics.utils.metrics.ClassifyMetrics object with attributes:

confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x791f2df74080>
curves: []
curves_results: []
fitness: 0.9482758641242981
keys: ['metrics/accuracy_top1', 'metrics/accuracy_top5']
results_dict: {'metrics/accuracy_top1': 0.8965517282485962, 'metrics/accuracy_top5': 1.0, 'fitness': 0.9482758641242981}
save_dir: PosixPath('/content/drive/MyDrive/Colab Notebooks/Yolo-cls-solar-plane/runs/classify/train10')
speed: {'preprocess': 0.0765177011486941, 'inference': 0.21266158620718847, 'loss': 0.00013245977013072148, 'postprocess': 0.00026983908073502946}
task: 'classify'
top1: 0.8965517282485962
top5: 1.0

In [ ]:
# ปลด freeze ทั้งโมเดล
for p in model.model.parameters():
    p.requires_grad = True

if not hasattr(model, "overrides") or "model" not in model.overrides:
    model.overrides = getattr(model, "overrides", {})
    model.overrides["model"] = 'yolo11n-cls.pt'  # แค่เป็นสตริงบอกที่มาของโมเดล/คอนฟิก

# fine-tune ทั้งโมเดลต่อ
model.train(
    data=DATA_DIR,
    epochs=10,
    imgsz=224,
    batch=64,
    optimizer="Adam",
    lr0=5e-4,             # ลด LR ลงเพื่อปรับละเอียด
    momentum=0.9,
    weight_decay=5e-4
)

Ultralytics 8.3.214 🚀 Python-3.12.12 torch-2.8.0+cu126 CUDA:0 (Tesla T4, 15095MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=64, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=Faulty_solar_panel_split, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=10, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=224, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.0005, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11n-cls.pt, momentum=0.9, mosaic=1.0, multi_scale=False, name=train11, nbs=64, nms=False, opset=None, optimize=False, optimizer=Adam, overlap_mask=True, patience=100, perspective=0.0, plots=True, pose

ultralytics.utils.metrics.ClassifyMetrics object with attributes:

confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x791f2df853d0>
curves: []
curves_results: []
fitness: 0.9482758641242981
keys: ['metrics/accuracy_top1', 'metrics/accuracy_top5']
results_dict: {'metrics/accuracy_top1': 0.8965517282485962, 'metrics/accuracy_top5': 1.0, 'fitness': 0.9482758641242981}
save_dir: PosixPath('/content/drive/MyDrive/Colab Notebooks/Yolo-cls-solar-plane/runs/classify/train11')
speed: {'preprocess': 0.07483036781467355, 'inference': 0.24543024138156083, 'loss': 0.00011565517242757026, 'postprocess': 0.00022936781575460383}
task: 'classify'
top1: 0.8965517282485962
top5: 1.0

In [ ]:
model.val(
    data="Faulty_solar_panel_split",
    imgsz=224,
    batch=8,
    split="val",   # ค่าเริ่มต้นคือ "val"
)

Ultralytics 8.3.214 🚀 Python-3.12.12 torch-2.8.0+cu126 CUDA:0 (Tesla T4, 15095MiB)
YOLO11n-cls summary (fused): 47 layers, 1,532,429 parameters, 0 gradients, 3.2 GFLOPs
train: /content/drive/MyDrive/Colab Notebooks/Yolo-cls-solar-plane/Faulty_solar_panel_split/train... found 749 images in 5 classes ✅ 
val: /content/drive/MyDrive/Colab Notebooks/Yolo-cls-solar-plane/Faulty_solar_panel_split/val... found 87 images in 5 classes ✅ 
test: /content/drive/MyDrive/Colab Notebooks/Yolo-cls-solar-plane/Faulty_solar_panel_split/test... found 49 images in 5 classes ✅ 
val: Fast image access ✅ (ping: 0.6±0.2 ms, read: 124.3±153.9 MB/s, size: 448.5 KB)
val: Scanning /content/drive/MyDrive/Colab Notebooks/Yolo-cls-solar-plane/Faulty_solar_panel_split/val... 87 images, 0 corrupt: 100% ━━━━━━━━━━━━ 87/87 191.3Kit/s 0.0s
val: /content/drive/MyDrive/Colab Notebooks/Yolo-cls-solar-plane/Faulty_solar_panel_split/val/Dusty/Dust (17).jpg: corrupt JPEG restored and saved
               classes   top1_acc   to

ultralytics.utils.metrics.ClassifyMetrics object with attributes:

confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x791f1ff1a300>
curves: []
curves_results: []
fitness: 0.9482758641242981
keys: ['metrics/accuracy_top1', 'metrics/accuracy_top5']
results_dict: {'metrics/accuracy_top1': 0.8965517282485962, 'metrics/accuracy_top5': 1.0, 'fitness': 0.9482758641242981}
save_dir: PosixPath('/content/drive/MyDrive/Colab Notebooks/Yolo-cls-solar-plane/runs/classify/val6')
speed: {'preprocess': 0.11096148275895361, 'inference': 4.1639024827637865, 'loss': 0.0012930114782397264, 'postprocess': 0.0019218735611377334}
task: 'classify'
top1: 0.8965517282485962
top5: 1.0

In [ ]:
modle = YOLO('runs/classify/train11/weights/best.pt')
model.val(
    data="Faulty_solar_panel_split",
    imgsz=224,
    batch=8,
    split="test",   # ค่าเริ่มต้นคือ "val"
)

Ultralytics 8.3.214 🚀 Python-3.12.12 torch-2.8.0+cu126 CUDA:0 (Tesla T4, 15095MiB)
train: /content/drive/MyDrive/Colab Notebooks/Yolo-cls-solar-plane/Faulty_solar_panel_split/train... found 749 images in 5 classes ✅ 
val: /content/drive/MyDrive/Colab Notebooks/Yolo-cls-solar-plane/Faulty_solar_panel_split/val... found 87 images in 5 classes ✅ 
test: /content/drive/MyDrive/Colab Notebooks/Yolo-cls-solar-plane/Faulty_solar_panel_split/test... found 49 images in 5 classes ✅ 
test: Fast image access ✅ (ping: 0.5±0.2 ms, read: 46.0±39.0 MB/s, size: 89.7 KB)
test: Scanning /content/drive/MyDrive/Colab Notebooks/Yolo-cls-solar-plane/Faulty_solar_panel_split/test... 49 images, 0 corrupt: 100% ━━━━━━━━━━━━ 49/49 97.9Kit/s 0.0s
               classes   top1_acc   top5_acc: 100% ━━━━━━━━━━━━ 7/7 3.4it/s 2.1s
                   all      0.918          1
Speed: 0.7ms preprocess, 2.5ms inference, 0.0ms loss, 0.0ms postprocess per image
Results saved to /content/drive/MyDrive/Colab Notebooks/Yolo-cls

ultralytics.utils.metrics.ClassifyMetrics object with attributes:

confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x791ee3543680>
curves: []
curves_results: []
fitness: 0.9591836631298065
keys: ['metrics/accuracy_top1', 'metrics/accuracy_top5']
results_dict: {'metrics/accuracy_top1': 0.918367326259613, 'metrics/accuracy_top5': 1.0, 'fitness': 0.9591836631298065}
save_dir: PosixPath('/content/drive/MyDrive/Colab Notebooks/Yolo-cls-solar-plane/runs/classify/val8')
speed: {'preprocess': 0.6539223061420727, 'inference': 2.481449285712689, 'loss': 0.004946142854926305, 'postprocess': 0.0025524489835649492}
task: 'classify'
top1: 0.918367326259613
top5: 1.0

In [ ]:
import os
from pathlib import Path
import cv2
import numpy as np


# ================== CONFIG ==================
MODEL_PATH = "runs/classify/train11/weights/best.pt"  #
TEST_DIR   = Path("Faulty_solar_panel_split/test")
OUT_DIR    = Path("runs/cls_test_vis")   # โฟลเดอร์เซฟภาพพร้อม overlay
TOPK       = 5          # แสดง top-k
SAVE_OUT   = True       # True = บันทึกไฟล์ overlay
SHOW_N     = 16         # แสดงรูปกี่รูปในโน้ตบุ๊ก (ตั้ง None เพื่อไม่แสดง)
N_COLS     = 4          # จำนวนคอลัมน์เวลาวางภาพในกริด
IMG_SIZE   = 224        # ขนาดอินพุตให้โมเดล
# ============================================

def get_image_paths(root: Path):
    exts = (".jpg",".jpeg",".png",".bmp",".webp",".tif",".tiff")
    return sorted([p for p in root.rglob("*") if p.suffix.lower() in exts])

def idx2name_map(names):
    # รองรับทั้ง list/dict ตามเวอร์ชันของ ultralytics
    if isinstance(names, dict):
        return names
    return {i: n for i, n in enumerate(names)}

def draw_text_lines(img_bgr, lines, x=10, y=10, font=cv2.FONT_HERSHEY_SIMPLEX,
                    scale=0.6, color=(255,255,255), thick=2, bg=(0,0,0), alpha=0.5, line_gap=24):
    """วาดหลายบรรทัดพร้อมพื้นหลังโปร่งใสให้อ่านง่าย"""
    for line in lines:
        (w, h), base = cv2.getTextSize(line, font, scale, thick)
        overlay = img_bgr.copy()
        cv2.rectangle(overlay, (x-3, y-3), (x + w + 6, y + h + 6), bg, -1)
        cv2.addWeighted(overlay, alpha, img_bgr, 1 - alpha, 0, img_bgr)
        cv2.putText(img_bgr, line, (x, y + h), font, scale, color, thick, cv2.LINE_AA)
        y += line_gap
    return img_bgr

def predict_one_image(model, img_bgr, topk=5, imgsz=224):
    """รับภาพ BGR -> ทำนายคลาส -> คืน (indices, confs) ยาวไม่เกิน topk"""
    rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    results = model.predict(rgb, imgsz=imgsz, verbose=False)
    res = results[0]
    probs = getattr(res, "probs", None)
    if probs is None or probs.data is None:
        return None, None
    # ถ้ามี top5/top5conf ก็ใช้เลย ไม่งั้นคัดเอง
    if hasattr(probs, "top5") and hasattr(probs, "top5conf"):
        idxs = probs.top5[:topk]
        confs = probs.top5conf[:topk]
    else:
        p = probs.data.detach().cpu().numpy().flatten()
        idxs = p.argsort()[-topk:][::-1]
        confs = p[idxs]
    return idxs, confs


print("[INFO] Loading model...")
model_test = YOLO(MODEL_PATH)
class_names = idx2name_map(model.names)

test_images = get_image_paths(TEST_DIR)
print(f"[INFO] Found {len(test_images)} images in test/")
if len(test_images) == 0:
    raise SystemExit("No test images found. Check TEST_DIR.")

[INFO] Loading model...
[INFO] Found 49 images in test/


In [ ]:
from tqdm import tqdm
vis_paths = []  # เก็บพาธไฟล์ที่บันทึกแล้ว เผื่อใช้ต่อ
for img_path in tqdm(test_images, desc="Predict & save overlays"):
    bgr = cv2.imread(str(img_path))
    if bgr is None:
        print(f"[WARN] Cannot read: {img_path}")
        continue

    idxs, confs = predict_one_image(model, bgr, topk=TOPK, imgsz=IMG_SIZE)
    if idxs is None:
        print(f"[WARN] No probabilities for: {img_path}")
        continue

    # เตรียมข้อความ
    lines = []
    for rank, (idx, cf) in enumerate(zip(idxs, confs), start=1):
        cls_name = class_names.get(int(idx), str(idx))
        lines.append(f"{rank}. {cls_name}: {float(cf)*100:.2f}%")

    # วาด overlay
    bgr_out = bgr.copy()
    bgr_out = draw_text_lines(bgr_out, lines, x=10, y=10, scale=0.6, thick=2,
                              color=(255,255,255), bg=(0,0,0), alpha=0.5, line_gap=24)

    if SAVE_OUT:
        rel = img_path.relative_to(TEST_DIR)
        save_path = OUT_DIR / rel
        save_path.parent.mkdir(parents=True, exist_ok=True)
        cv2.imwrite(str(save_path), bgr_out)
        vis_paths.append(save_path)

print(f"[DONE] Saved {len(vis_paths)} overlays to: {OUT_DIR.resolve()}" if SAVE_OUT else "[DONE] Done.")


Predict & save overlays: 100%|██████████| 49/49 [00:06<00:00,  7.80it/s]

[DONE] Saved 49 overlays to: /content/drive/MyDrive/Colab Notebooks/Yolo-cls-solar-plane/runs/cls_test_vis


In [ ]:
import matplotlib.pyplot as plt
# แสดงตัวอย่างภาพ overlay ในกริด (อ่านจากไฟล์ที่เพิ่งบันทึก)
if SHOW_N:
    to_show = vis_paths[:SHOW_N] if SAVE_OUT else test_images[:SHOW_N]
    n = len(to_show)
    n_cols = N_COLS
    n_rows = int(np.ceil(n / n_cols))
    plt.figure(figsize=(4*n_cols, 4*n_rows))
    for i, p in enumerate(to_show, 1):
        img_bgr = cv2.imread(str(p)) if SAVE_OUT else cv2.imread(str(p))
        if img_bgr is None:
            continue
        img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
        ax = plt.subplot(n_rows, n_cols, i)
        ax.imshow(img_rgb)
        ax.set_title(str(Path(p).name), fontsize=10)
        ax.axis("off")
    plt.tight_layout()
    plt.show()

Output hidden; open in https://colab.research.google.com to view.

In [ ]:
%%writefile streamlit_app.py
import streamlit as st
from ultralytics import YOLO
import cv2
import numpy as np
from PIL import Image

# Load the best model
MODEL_PATH = "runs/classify/train11/weights/best.pt"
model = YOLO(MODEL_PATH)
class_names = model.names # Get class names from the model

# Function to predict on an image
def predict_image(image, model, imgsz=224):
    results = model.predict(image, imgsz=imgsz, verbose=False)
    res = results[0]
    probs = getattr(res, "probs", None)
    if probs is None or probs.data is None:
        return None, None

    p = probs.data.detach().cpu().numpy().flatten()
    idxs = p.argsort()[-5:][::-1] # Get top 5
    confs = p[idxs]
    return idxs, confs

st.title("Solar Panel Fault Classification")

st.sidebar.title("Input Options")
input_mode = st.sidebar.radio("Select input mode:", ("Upload Image", "Webcam"))

if input_mode == "Upload Image":
    uploaded_file = st.sidebar.file_uploader("Upload an image...", type=["jpg", "jpeg", "png"])

    if uploaded_file is not None:
        # Read the image
        image = Image.open(uploaded_file)
        img_np = np.array(image) # Convert PIL image to numpy array
        img_bgr = cv2.cvtColor(img_np, cv2.COLOR_RGB2BGR) # Convert RGB to BGR for OpenCV

        st.image(image, caption="Uploaded Image.", use_column_width=True)
        st.write("")
        st.write("Classifying...")

        idxs, confs = predict_image(img_bgr, model)

        if idxs is not None:
            st.subheader("Prediction Results:")
            for rank, (idx, cf) in enumerate(zip(idxs, confs), start=1):
                cls_name = class_names.get(int(idx), str(idx))
                st.write(f"{rank}. {cls_name}: {float(cf)*100:.2f}%")
        else:
            st.write("Could not get prediction probabilities.")

elif input_mode == "Webcam":
    st.subheader("Webcam Feed")
    st.write("Please allow webcam access in your browser.")

    # You would typically use a Streamlit component for webcam capture.
    # For a simple example, you can inform the user about this.
    st.warning("Webcam functionality requires a specific Streamlit component or deployment setup.")
    st.info("For local development, you might need to run `streamlit run your_app.py` from your terminal after installing the necessary libraries.")

    # A placeholder for potential future webcam integration
    # img_file_buffer = st.camera_input("Take a picture")
    # if img_file_buffer is not None:
    #     # To read image file buffer as a PIL Image:
    #     img = Image.open(img_file_buffer)
    #     img_np = np.array(img)
    #     img_bgr = cv2.cvtColor(img_np, cv2.COLOR_RGB2BGR)

    #     st.image(img, caption="Captured Image.", use_column_width=True)
    #     st.write("Classifying...")

    #     idxs, confs = predict_image(img_bgr, model)

    #     if idxs is not None:
    #         st.subheader("Prediction Results:")
    #         for rank, (idx, cf) in enumerate(zip(idxs, confs), start=1):
    #             cls_name = class_names.get(int(idx), str(idx))
    #             st.write(f"{rank}. {cls_name}: {float(cf)*100:.2f}%")
    #     else:
    #         st.write("Could not get prediction probabilities.")